# Решения: fit и порог

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def _find(name: str) -> Path:
    for p in (Path(name), Path(f'../../data/{name}'), Path(f'../data/{name}')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'{name} не найден рядом с ноутбуком')


CSV_PATH = _find('bank_marketing_slim.csv')
df = pd.read_csv(CSV_PATH)
target = (df['y'] == 'yes').astype(int)

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score


In [ ]:
feature_columns = [c for c in df.columns if c not in ('y', 'duration')]
X_raw = pd.get_dummies(df[feature_columns], drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, target, test_size=0.25, random_state=61, stratify=target
)
model = LogisticRegression(max_iter=1200)
model.fit(X_train, y_train)
proba_test = model.predict_proba(X_test)[:, 1]
rows = []
for thr in (0.3, 0.5, 0.7):
    pred = (proba_test >= thr).astype(int)
    rows.append(
        {
            'threshold': thr,
            'precision': float(precision_score(y_test, pred, zero_division=0)),
            'recall': float(recall_score(y_test, pred, zero_division=0)),
            'f1': float(f1_score(y_test, pred, zero_division=0)),
        }
    )
table = pd.DataFrame(rows)
best_threshold = None
best_row = None
for row in rows:
    if row['recall'] >= 0.70:
        best_threshold = row['threshold']
        best_row = row
        break
if best_threshold is None:
    best_threshold = rows[0]['threshold']
    best_row = rows[0]
TEAM_NOTE = (
    f'Для пилота выбираем threshold={best_threshold:.2f}: recall={best_row['recall']:.3f}, '
    f'precision={best_row['precision']:.3f}. Это снижает риск пропуска заинтересованных клиентов '
    'при контролируемом росте ложных срабатываний.'
)
print(table)
print('best_threshold=', best_threshold)
print(TEAM_NOTE)